<a href="https://colab.research.google.com/github/JoshuaBL2002/Advanced-Data-Science-Project-Repository/blob/main/notebooks/Air_Quality_NO2_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Air Quality NO₂ Prediction Project

## 1. Data Loading and Exploration

Dataset: Low-cost air quality sensors and reference monitoring stations in Rouen, France.

Objective: Investigate whether machine learning models can predict reference NO₂ concentrations using low-cost sensor measurements.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:

# Define the target zip path directly to avoid relying on the interactive 'uploaded' variable
zip_filename = "/content/Air quality low-cost sensors and monitoring stations NO2 raw dataset in Rouen (France).zip"
extract_destination = "/content/rouen_dataset"

# Extract zip file automatically
if os.path.exists(zip_filename):
    print(f"Extracting {os.path.basename(zip_filename)}...")
    with zipfile.ZipFile(zip_filename, "r") as zip_ref:
        zip_ref.extractall(extract_destination)

    # Walk through directory to find and list extracted CSVs
    print("Extracted CSV files:")
    for root, dirs, filenames in os.walk(extract_destination):
        for filename in filenames:
            if filename.endswith(".csv"):
                print(os.path.join(root, filename))
else:
    print(f"Dataset zip file not found at: {zip_filename}")

In [ ]:
import os
import zipfile
import pandas as pd

zip_filename = "/content/Air quality low-cost sensors and monitoring stations NO2 raw dataset in Rouen (France).zip"
extract_destination = "/content/rouen_dataset"

# Ensure extraction is performed
if os.path.exists(zip_filename) and not os.path.exists(extract_destination):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_filename, "r") as zip_ref:
        zip_ref.extractall(extract_destination)

# Set correct path to ASE4.csv
ase4_path = "/content/rouen_dataset/Air quality low-cost sensors and monitoring stations NO2 raw dataset in Rouen (France)/ASE4.csv"

if not os.path.exists(ase4_path):
    # Fallback search if path structure differs
    for root, dirs, filenames in os.walk(extract_destination):
        for filename in filenames:
            if filename == "ASE4.csv":
                ase4_path = os.path.join(root, filename)
                break

print(f"Loading file from: {ase4_path}")
df = pd.read_csv(ase4_path, sep=";")
display(df.head())

In [ ]:
# Print the dimensions (rows, columns) of the loaded ASE4 DataFrame
print("Dataset dimensions:", df.shape)

# List all columns present in the dataset
print("\nColumn names:")
print(df.columns.tolist())

# Display data types, memory usage, and non-null counts per column
print("\nData types:")
df.info()

# Display the count of missing (NaN) values for each column
print("\nMissing values:")
display(df.isnull().sum())

# 2. Data Preprocessing

This section prepares the air quality sensor data for
machine learning by examining missing values, checking
duplicates, converting data types and creating a single
reference NO₂ target variable.

In [ ]:
# Check missing values in each column
missing_values = df.isnull().sum()

# Calculate percentage of missing values
missing_percentage = (missing_values / len(df)) * 100

# Display results
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage (%)": missing_percentage
})

display(missing_summary)

In [ ]:
# Count duplicate rows
duplicates = df.duplicated().sum()

print("Number of duplicate rows:", duplicates)

In [ ]:
# Display column names
print(df.columns.tolist())

# Display the first five records
display(df.head())

In [ ]:
# Create a working copy
df_clean = df.copy()

# Create the dependent variable
df_clean["Reference_NO2"] = np.where(
    df_clean["Location"] == "SUD3",
    df_clean["SUD3_NO2"],
    np.where(
        df_clean["Location"] == "QDP",
        df_clean["QDP_NO2"],
        np.nan
    )
)

# 1. Handle missing target values: Drop rows where Reference_NO2 is missing since we cannot use them for supervised learning
initial_len = len(df_clean)
df_clean = df_clean.dropna(subset=["Reference_NO2"])
print(f"Dropped {initial_len - len(df_clean)} rows due to missing Reference_NO2 target values.")

# 2. Handle duplicates
duplicates_count = df_clean.duplicated().sum()
if duplicates_count > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"Dropped {duplicates_count} duplicate rows.")
else:
    print("No duplicate rows found to drop.")

# Display the cleaned result
display(df_clean[["Location", "Reference_NO2"]].head(10))

In [ ]:
# 3. Check and validate Date/Time Conversion
# Inspect the date parsing process and handle any parsing failures explicitly
df_clean['DateTime'] = pd.to_datetime(df_clean['DateTime'], errors='coerce')
failed_dates = df_clean['DateTime'].isnull().sum()
print(f"Number of timestamps that failed to parse: {failed_dates}")
if failed_dates > 0:
    df_clean = df_clean.dropna(subset=['DateTime'])
    print(f"Dropped {failed_dates} rows with invalid date formats.")

# 3. Exploratory Data Analysis (EDA)

This section investigates patterns and relationships in the
air quality sensor data using statistical summaries and
visualisations.

The analysis focuses on NO₂ concentrations, differences
between monitoring stations, and relationships between
sensor measurements and reference NO₂ concentrations.

In [ ]:
### 3.1 Distribution of Reference NO₂
# Calculate and display descriptive statistics (mean, median, standard deviation, and range)
# for the compiled reference NO₂ target variable.
display(df_clean["Reference_NO2"].describe())

# Visualize the frequency distribution of Reference_NO2 using a histogram.
# The Kernel Density Estimate (KDE) curve helps identify skewness and overall shape.
plt.figure(figsize=(10, 5))
sns.histplot(
    data=df_clean,
    x="Reference_NO2",
    bins=30,
    kde=True
)

plt.title("Distribution of Reference NO₂ Concentrations")
plt.xlabel("Reference_NO2 (µg/m³)")
plt.ylabel("Frequency")
plt.show()

### Observation & Discussion:
- **Distribution of Reference NO₂**: The distribution of reference NO₂ concentration is right-skewed, showing a high density of measurements concentrated around low to moderate values (between 10 µg/m³ and 35 µg/m³).
- **Outliers**: There is a long tail extending up to 100+ µg/m³, which represents periods of high air pollution or transient local emission spikes. These outliers are critical for machine learning since capturing extreme events is an important capability for environmental prediction models.

In [ ]:
### 3.2 Comparison Across Stations
# Use a box plot to examine the distribution of the target variable grouped by physical location.
# This visualizes station-specific medians, quartiles, and potential outliers.
plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df_clean,
    x="Location",
    y="Reference_NO2"
)

plt.title("Reference NO₂ by Monitoring Station")
plt.xlabel("Monitoring Station")
plt.ylabel("Reference NO₂ Concentration")
plt.show()

### Observation & Discussion:
- **Comparison by Monitoring Station**: The boxplot shows the overall median concentration of NO₂ is slightly higher at the QDP station compared to SUD3.
- **Variability**: QDP displays a wider interquartile range (IQR), suggesting more fluctuating conditions, while SUD3 features a more compressed distribution with several high-value outliers.

In [ ]:
### 3.3 Correlation Analysis
# Filter for numerical columns to measure linear relationships between variables.
numeric_data = df_clean.select_dtypes(include=["number"])

# Compute Pearson correlation coefficients.
correlation_matrix = numeric_data.corr()

# Plot a annotated heatmap to highlight dependencies between sensor nodes and target values.
plt.figure(figsize=(12, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix of Numerical Variables")
plt.show()

### Observation & Discussion:
- **Full Correlation Analysis**: Highly strong correlations are observed between co-located measurements.
- **Target Correlation**: The new heatmap focuses strictly on potential features (predictors) like the raw sensor voltages/resistances, temperature, and relative humidity against the target `Reference_NO2` variable. This prevents leakage of unaligned reference station variables.

In [ ]:
### 3.3.1 Predictor-Only Correlation Heatmap
# Filter columns to isolate only intended model predictors (e.g., sensor inputs, temperature, humidity) and the target variable
predictor_cols = [col for col in numeric_data.columns if not any(x in col for x in ['SUD3_NO2', 'QDP_NO2', 'SUD3_O3', 'QDP_O3']) or col == 'Reference_NO2']

plt.figure(figsize=(10, 6))
sns.heatmap(
    numeric_data[predictor_cols].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Predictor vs Target Correlation Matrix")
plt.show()

In [ ]:
### 3.4 Temporal Trends
# Explicitly convert DateTime column to pandas datetime objects to prevent sorting and formatting errors
df_clean['DateTime'] = pd.to_datetime(df_clean['DateTime'], errors='coerce')

# Sort the dataset chronologically
df_time = df_clean.dropna(subset=['DateTime']).sort_values('DateTime')

# Draw the continuous trend of Reference_NO2 over time
plt.figure(figsize=(12, 5))
sns.lineplot(
    data=df_time,
    x="DateTime",
    y="Reference_NO2"
)

plt.title("Reference NO₂ Concentrations Over Time")
plt.xlabel("Date and Time")
plt.ylabel("Reference NO₂ Concentration")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
### 3.5 Multi-Sensor Comparison (ASE4, ASE5, ASE6)
# Load additional datasets to establish comparisons across multiple low-cost sensors
ase_datasets = {}
for sensor_name in ["ASE4", "ASE5", "ASE6"]:
    # Try to find file paths dynamically in extraction destination
    file_path = None
    for root, dirs, filenames in os.walk(extract_destination):
        for filename in filenames:
            if filename == f"{sensor_name}.csv":
                file_path = os.path.join(root, filename)
                break
    if file_path:
        print(f"Loading {sensor_name} from {file_path}")
        temp_df = pd.read_csv(file_path, sep=";")
        temp_df['Sensor_ID'] = sensor_name
        # Standardize target alignment inside each sensor
        temp_df["Reference_NO2"] = np.where(
            temp_df["Location"] == "SUD3",
            temp_df["SUD3_NO2"],
            np.where(temp_df["Location"] == "QDP", temp_df["QDP_NO2"], np.nan)
        )
        ase_datasets[sensor_name] = temp_df.dropna(subset=["Reference_NO2"])

# Combine datasets for comparative visualization
if len(ase_datasets) > 0:
    combined_df = pd.concat(ase_datasets.values(), ignore_index=True)
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=combined_df, x="Sensor_ID", y="Reference_NO2", hue="Location")
    plt.title("Comparison of Reference NO₂ across Sensors ASE4, ASE5, and ASE6")
    plt.ylabel("Reference NO₂ Concentration (µg/m³)")
    plt.show()
else:
    print("Could not locate datasets for ASE5 and ASE6 for comparison.")